In [1]:
!pip install -q -U \
    langchain langchain-community langchain-huggingface langchain-text-splitters \
    faiss-cpu \
    pypdf \
    gspread google-auth \
    transformers accelerate bitsandbytes \
    sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.8/256.8 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


CUDA available: True
Device: Tesla T4


In [3]:
import os
import getpass
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline, ChatHuggingFace
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

try:
    from langchain.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
except ImportError:
    from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain


/tmp/ipykernel_510/2958462675.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
from google.colab import files

print("Upload one or more PDF files (your knowledge base).")
uploaded = files.upload()
pdf_paths = list(uploaded.keys())
print("Uploaded:", pdf_paths)


Upload one or more PDF files (your knowledge base).


Saving 1706.03762v7.pdf to 1706.03762v7.pdf
Uploaded: ['1706.03762v7.pdf']


In [5]:
pdf_documents = []

for path in pdf_paths:
    loader = PyPDFLoader(path)
    pages = loader.load()
    pdf_documents.extend(pages)

print(f"Loaded {len(pdf_documents)} pages from {len(pdf_paths)} PDF(s)")
if pdf_documents:
    print(pdf_documents[0].page_content[:300])


Loaded 15 pages from 1 PDF(s)
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par


In [7]:

USE_GSPREAD = True
SHEET_URL = "PASTE_YOUR_GOOGLE_SHEET_URL_HERE"

sheet_rows = []

if USE_GSPREAD:
    from google.colab import auth
    auth.authenticate_user()

    import gspread
    from google.auth import default

    creds, _ = default()
    gc = gspread.authorize(creds)

    worksheet = gc.open_by_url("https://docs.google.com/spreadsheets/d/1x2ECTwDCBHUby_IIGJd_4auWyc9B8_eP3IbW5CK8Se4/edit?usp=drive_link").sheet1
    sheet_rows = worksheet.get_all_records()
    print(f"Loaded {len(sheet_rows)} rows from Google Sheet")


Loaded 20 rows from Google Sheet


In [8]:
sheet_documents = []

for i, row in enumerate(sheet_rows):

    text = "\n".join(f"{k}: {v}" for k, v in row.items())
    sheet_documents.append(
        Document(page_content=text, metadata={"source": "google_sheet", "row": i})
    )

print(f"Converted {len(sheet_documents)} sheet rows into documents")


Converted 20 sheet rows into documents


In [9]:

all_documents = pdf_documents + sheet_documents
print(f"Total documents before splitting: {len(all_documents)}")


Total documents before splitting: 35


In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(all_documents)
print(f"Split into {len(chunks)} chunks")


Split into 86 chunks


In [11]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = FAISS.from_documents(chunks, embedding_model)
vectorstore.save_local("faiss_index")
print("FAISS index built and saved to ./faiss_index")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index built and saved to ./faiss_index


In [12]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

# quick sanity check
test_results = retriever.invoke("What is this document about?")
for doc in test_results:
    print("-", doc.metadata.get("source"), "|", doc.page_content[:120].replace("\n", " "))


- google_sheet | Topic: Space Missions Description: Apollo, Voyager, Cassini, Perseverance.
- google_sheet | Topic: The Universe Description: The universe contains all space, time, matter, and energy.
- google_sheet | Topic: Solar System Description: The Sun, eight planets, dwarf planets, moons, asteroids, and comets.
- google_sheet | Topic: Stars Description: Massive luminous spheres of plasma.


## 3. Question Answering Pipeline (RAG)

Load a local LLM on the T4 GPU (4-bit quantized so **Qwen2.5-3B-Instruct** comfortably fits), then wire it into LangChain as a chat model.

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.3,
    repetition_penalty=1.1,
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=gen_pipeline)
chat_model = ChatHuggingFace(llm=llm)
print("LLM ready:", MODEL_ID)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'repetition_penalty', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM ready: Qwen/Qwen2.5-3B-Instruct


In [14]:

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question, which might reference "
    "context in the chat history, formulate a standalone question that can be "
    "understood without the chat history. Do NOT answer the question, just "
    "reformulate it if needed, otherwise return it as is."
)
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(
    chat_model, retriever, contextualize_q_prompt
)


In [15]:

qa_system_prompt = (
    "You are a helpful knowledge assistant. Use ONLY the following retrieved "
    "context to answer the user's question. If the answer isn't in the "
    "context, say you don't know -- do not make things up.\n\n"
    "Context:\n{context}"
)
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(chat_model, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)
print("RAG chain ready")


RAG chain ready


## 4. Memory Integration



In [16]:

session_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)
print("Conversational RAG chain with memory ready")


Conversational RAG chain with memory ready


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [17]:
def ask(question: str, session_id: str = "default"):
    """Ask the assistant a question, keeping memory for the given session_id."""
    result = conversational_rag_chain.invoke(
        {"input": question},
        config={"configurable": {"session_id": session_id}},
    )
    return result["answer"], result["context"]


## 5. Testing & Evaluation



In [18]:
def ask_and_show(question, session_id="default"):
    answer, context = ask(question, session_id)
    print(f"Q: {question}")
    print(f"A: {answer}\n")
    print("Retrieved context:")
    for doc in context:
        preview = doc.page_content[:150].replace("\n", " ")
        print(f"  [{doc.metadata.get('source')}] {preview}...")
    print("-" * 80)
    return answer


ask_and_show("According to the Attention Is All You Need paper, why is dot-product attention faster in practice than additive attention?")


ask_and_show("What happens to those dot products for large values of dk if they are not scaled?")

ask_and_show("Switching to astronomy, what are the major telescopes and space missions listed in the database?")


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) se

Q: According to the Attention Is All You Need paper, why is dot-product attention faster in practice than additive attention?
A: According to the Attention Is All You Need paper, dot-product attention is much faster and more space-efficient in practice because it can be implemented using highly optimized matrix multiplication code. Additionally, the paper suggests that for large values of dk, dot-product attention performs better due to scaling the dot products by \( \frac{1}{\sqrt{d_k}} \), which helps mitigate the issue of extremely small gradients in the softmax function. However, the primary reason given for its speed and efficiency is its implementation through matrix operations.

Retrieved context:
  [1706.03762v7.pdf] much faster and more space-efficient in practice, since it can be implemented using highly optimized matrix multiplication code. While for small value...
  [1706.03762v7.pdf] values. In practice, we compute the attention function on a set of queries simultaneously,

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/e

Q: What happens to those dot products for large values of dk if they are not scaled?
A: If those dot products for large values of dk are not scaled by \( \frac{1}{\sqrt{d_k}} \), they can grow very large in magnitude. This can cause the softmax function to have extremely small gradients, making it difficult to optimize effectively during training. The scaling helps to normalize these dot products so that the softmax function operates within a range where it can provide meaningful gradient information.

Retrieved context:
  [1706.03762v7.pdf] much faster and more space-efficient in practice, since it can be implemented using highly optimized matrix multiplication code. While for small value...
  [1706.03762v7.pdf] we found it beneficial to linearly project the queries, keys and values h times with different, learned linear projections to dk, dk and dv dimensions...
  [1706.03762v7.pdf] Scaled Dot-Product Attention  Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (rig

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Q: Switching to astronomy, what are the major telescopes and space missions listed in the database?
A: Based on the provided context, the major telescopes mentioned are:

- **Hubble**: A space telescope operated by NASA and the European Space Agency.
- **James Webb**: An upcoming infrared telescope also operated by NASA and the European Space Agency, scheduled to launch in 2021.

The space missions listed are:

- **Apollo**: A series of U.S. lunar exploration missions conducted from 1968 to 1972.
- **Voyager**: A series of robotic spacecraft sent to explore the outer Solar System, launched in 1977.
- **Cassini**: A mission to orbit Saturn and its moons, launched in 1997.
- **Perseverance**: A Mars rover currently exploring Mars, launched in 2020.

Retrieved context:
  [google_sheet] Topic: Major Telescopes Description: Hubble, James Webb, ALMA....
  [google_sheet] Topic: Space Missions Description: Apollo, Voyager, Cassini, Perseverance....
  [google_sheet] Topic: Galaxies Description:

'Based on the provided context, the major telescopes mentioned are:\n\n- **Hubble**: A space telescope operated by NASA and the European Space Agency.\n- **James Webb**: An upcoming infrared telescope also operated by NASA and the European Space Agency, scheduled to launch in 2021.\n\nThe space missions listed are:\n\n- **Apollo**: A series of U.S. lunar exploration missions conducted from 1968 to 1972.\n- **Voyager**: A series of robotic spacecraft sent to explore the outer Solar System, launched in 1977.\n- **Cassini**: A mission to orbit Saturn and its moons, launched in 1997.\n- **Perseverance**: A Mars rover currently exploring Mars, launched in 2020.'

In [19]:
# Fresh session -> memory should NOT carry over from the chat above
ask_and_show("What did I just ask you?", session_id="new_session")


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Q: What did I just ask you?
A: You didn't ask me anything specific based on the given context. The context provided descriptions for topics like galaxies, Earth, the universe, and Jupiter but didn't include questions. So, I don't know what you asked about your query.

Retrieved context:
  [google_sheet] Topic: Galaxies Description: Huge collections of stars, gas, dust, and dark matter....
  [google_sheet] Topic: Earth Description: Only known planet with life....
  [google_sheet] Topic: The Universe Description: The universe contains all space, time, matter, and energy....
  [google_sheet] Topic: Jupiter Description: Largest planet....
--------------------------------------------------------------------------------


"You didn't ask me anything specific based on the given context. The context provided descriptions for topics like galaxies, Earth, the universe, and Jupiter but didn't include questions. So, I don't know what you asked about your query."

In [20]:

for msg in session_store["default"].messages:
    print(f"{msg.type}: {msg.content[:500]}")


human: According to the Attention Is All You Need paper, why is dot-product attention faster in practice than additive attention?
ai: According to the Attention Is All You Need paper, dot-product attention is much faster and more space-efficient in practice because it can be implemented using highly optimized matrix multiplication code. Additionally, the paper suggests that for large values of dk, dot-product attention performs better due to scaling the dot products by \( \frac{1}{\sqrt{d_k}} \), which helps mitigate the issue of extremely small gradients in the softmax function. However, the primary reason given for its speed
human: What happens to those dot products for large values of dk if they are not scaled?
ai: If those dot products for large values of dk are not scaled by \( \frac{1}{\sqrt{d_k}} \), they can grow very large in magnitude. This can cause the softmax function to have extremely small gradients, making it difficult to optimize effectively during training. The scalin

### Manual evaluation checklist

For each test question above, check:
- **Relevance** — did the retrieved chunks actually relate to the question?
- **Accuracy** — is the answer grounded in the retrieved context (no hallucination)?
- **Memory** — did follow-up questions correctly use prior turns, and did the `new_session`
  test confirm sessions don't leak into each other?

Add more `ask_and_show(...)` calls above with your own questions to keep testing.